# LLM-as-Judge Scoring

Reads evaluation results from `output/evaluation_results.csv` and scores each row with gpt-5-mini.

Saves versioned output to `output/judge/judged_v{N}.csv`.

Run this notebook multiple times (3×) to enable ICC reliability analysis.

In [ ]:
import pandas as pd
import json
import glob
import os
from pathlib import Path
from tqdm.auto import tqdm
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

from config import settings, JUDGE_MODEL, OUTPUT_PATH, PIPELINE_LABELS, SEED

LIST_COLUMNS = ["context", "retrieved_doc_contents"]
JUDGE_OUTPUT_DIR = "output/judge"


def parse_list_columns(df):
    for col in LIST_COLUMNS:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: json.loads(x) if isinstance(x, str) and x.strip() else []
            )
    return df


def serialize_list_columns(df):
    save_df = df.copy()
    for col in LIST_COLUMNS:
        save_df[col] = save_df[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else "[]"
        )
    return save_df


def add_pipeline_label(df):
    df["pipeline_label"] = df["pipeline"].map(PIPELINE_LABELS)
    return df

print("Imports OK")

In [ ]:
judge_llm = ChatOpenAI(
    model=JUDGE_MODEL["model_name"],
    temperature=JUDGE_MODEL["temperature"],
    api_key=settings.api_key,
    seed=JUDGE_MODEL.get("seed"),
)

JUDGE_PROMPT = """Bạn là giám khảo đánh giá chất lượng câu trả lời của chatbot tuyển sinh đại học.

CÂU HỎI:
{question}

CÂU TRẢ LỜI CHUẨN (ground truth):
{ground_truth}

CÂU TRẢ LỜI CỦA HỆ THỐNG:
{generated_answer}

NGỮ CẢNH TRUY XUẤT (nếu có):
{context}

Hãy chấm điểm (1-5) cho 3 tiêu chí sau:
1. **Correctness**: Câu trả lời có chính xác so với ground truth không?
2. **Completeness**: Câu trả lời có đầy đủ thông tin không?
3. **Faithfulness**: Câu trả lời có trung thực với ngữ cảnh truy xuất không? (Nếu không có ngữ cảnh, chấm dựa trên ground truth)

Trả về JSON duy nhất, KHÔNG giải thích:
{{"correctness": <1-5>, "completeness": <1-5>, "faithfulness": <1-5>}}"""


def format_context(ctx):
    if isinstance(ctx, list) and ctx:
        return "\n---\n".join(ctx)
    return "(không có)"


async def judge_one(row: dict) -> dict:
    prompt = JUDGE_PROMPT.format(
        question=row["question"],
        ground_truth=row["ground_truth_answer"],
        generated_answer=row["generated_answer"],
        context=format_context(row.get("context")),
    )
    resp = await judge_llm.ainvoke([HumanMessage(content=prompt)])
    try:
        text = resp.content.strip()
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0]
        elif "```" in text:
            text = text.split("```")[1].split("```")[0]
        scores = json.loads(text.strip())
    except Exception:
        scores = {"correctness": 0, "completeness": 0, "faithfulness": 0}
    return scores


def next_judged_version() -> str:
    existing = sorted(glob.glob(f"{JUDGE_OUTPUT_DIR}/judged_v*.csv"))
    if not existing:
        return f"{JUDGE_OUTPUT_DIR}/judged_v1.csv"
    last = int(Path(existing[-1]).stem.split("_v")[1])
    return f"{JUDGE_OUTPUT_DIR}/judged_v{last + 1}.csv"


results_df = add_pipeline_label(parse_list_columns(pd.read_csv(OUTPUT_PATH)))
print(f"Scoring {len(results_df)} rows with judge model: {JUDGE_MODEL['model_name']}")

all_scores = []
for idx, row in tqdm(results_df.iterrows(), total=len(results_df), desc="Judging"):
    scores = await judge_one(row.to_dict())
    all_scores.append(scores)

scores_df = pd.DataFrame(all_scores)
results_df["correctness"] = scores_df["correctness"]
results_df["completeness"] = scores_df["completeness"]
results_df["faithfulness"] = scores_df["faithfulness"]

judged_path = next_judged_version()
os.makedirs(JUDGE_OUTPUT_DIR, exist_ok=True)
serialize_list_columns(results_df).to_csv(judged_path, index=False, encoding="utf-8-sig")
print(f"Saved scored results to {judged_path}")

In [ ]:
from scipy.stats import wilcoxon
import numpy as np
import pingouin as pg

judged_files = sorted(glob.glob(f"{JUDGE_OUTPUT_DIR}/judged_v*.csv"))
print(f"Found {len(judged_files)} judge runs: {judged_files}")

all_runs = [add_pipeline_label(parse_list_columns(pd.read_csv(f))) for f in judged_files]

metrics = ["correctness", "completeness", "faithfulness"]

# ── Main pipeline labels ──
main_labels = ["LLM Only", "Basic RAG", "Our RAG"]
main_keys   = ["llm_only", "basic_rag", "our_rag"]

# ── Ablation labels ──
ablation_labels = ["− Reranker", "− Query Expansion", "− Text2SQL", "− Tool Routing"]
ablation_keys   = ["ablation_no_rerank", "ablation_no_qe", "ablation_no_text2sql", "ablation_no_routing"]

all_labels = main_labels + ablation_labels
all_keys   = main_keys + ablation_keys

# ═══════════════════════════════════════════════════════════════════════
# TABLE 1: All Pipelines Comparison
# ═══════════════════════════════════════════════════════════════════════
print("=" * 70)
print("TABLE 1: Pipeline Comparison (mean ± std across judge runs)")
print("=" * 70)

rows = []
for label in all_labels:
    sub_runs = [run[run["pipeline_label"] == label] for run in all_runs]
    if sub_runs[0].empty:
        continue
    row_data = {"pipeline": label}
    for m in metrics:
        vals = [s[m].mean() for s in sub_runs]
        row_data[m] = f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    row_data["latency"] = f"{sub_runs[0]['latency_s'].mean():.2f}"
    rows.append(row_data)

table1 = pd.DataFrame(rows).set_index("pipeline")
print(table1.to_markdown())

# ═══════════════════════════════════════════════════════════════════════
# TABLE 2: Score Distribution (main pipelines)
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("TABLE 2: Score Distribution (averaged across judge runs)")
print("=" * 70)

for label in main_labels:
    sub_runs = [run[run["pipeline_label"] == label] for run in all_runs]
    if sub_runs[0].empty:
        continue
    print(f"\n{label}:")
    for m in metrics:
        per_question = np.mean([s[m].values for s in sub_runs], axis=0)
        print(f"  {m}: mean={per_question.mean():.2f}, std={per_question.std():.2f}, "
              f"min={per_question.min():.1f}, max={per_question.max():.1f}, "
              f">=4: {(per_question >= 4).sum()}/{len(per_question)} "
              f"({(per_question >= 4).mean()*100:.0f}%)")

# ═══════════════════════════════════════════════════════════════════════
# TABLE 3: Latency
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("TABLE 3: Latency (seconds)")
print("=" * 70)

lat_cols = ["latency_s"]
if "latency_cached_s" in all_runs[0].columns:
    lat_cols.append("latency_cached_s")

lat = all_runs[0].groupby("pipeline_label").agg(
    cold_mean=("latency_s", "mean"),
    cold_median=("latency_s", "median"),
).round(3).reindex([l for l in all_labels if l in all_runs[0]["pipeline_label"].values])
print(lat.to_markdown())

# ═══════════════════════════════════════════════════════════════════════
# TABLE 4: Wilcoxon Signed-Rank Tests (main pipelines)
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("TABLE 4: Wilcoxon Signed-Rank Tests (paired, per-question)")
print("=" * 70)

avg_scores = {}
for label, key in zip(main_labels, main_keys):
    sub_runs = [run[run["pipeline_label"] == label] for run in all_runs]
    if sub_runs[0].empty:
        continue
    avg_scores[label] = {}
    for m in metrics:
        avg_scores[label][m] = np.mean([s[m].values for s in sub_runs], axis=0)

pairs = [
    ("Basic RAG", "LLM Only"),
    ("Our RAG", "LLM Only"),
    ("Our RAG", "Basic RAG"),
]

stat_rows = []
for a, b in pairs:
    if a not in avg_scores or b not in avg_scores:
        continue
    for m in metrics:
        diff = avg_scores[a][m] - avg_scores[b][m]
        if np.all(diff == 0):
            stat_rows.append({"comparison": f"{a} vs {b}", "metric": m,
                              "mean_diff": 0.0, "W": "-", "p": "-", "sig": "-"})
            continue
        w, p = wilcoxon(avg_scores[a][m], avg_scores[b][m])
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        stat_rows.append({
            "comparison": f"{a} vs {b}", "metric": m,
            "mean_diff": f"{diff.mean():.2f}", "W": f"{w:.0f}",
            "p": f"{p:.4f}", "sig": sig,
        })

stat_df = pd.DataFrame(stat_rows)
print(stat_df.to_markdown(index=False))

# ═══════════════════════════════════════════════════════════════════════
# TABLE 5: ICC
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print("TABLE 5: Inter-Run Agreement (ICC via pingouin)")
print("=" * 70)

if len(all_runs) >= 2:
    for m in metrics:
        icc_rows = []
        for run_idx, run in enumerate(all_runs):
            sub = run.sort_values(["pipeline", "qid"])[["pipeline", "qid", m]].copy()
            sub["rater"] = run_idx
            sub["target"] = range(len(sub))
            icc_rows.append(sub[["target", "rater", m]])

        icc_df = pd.concat(icc_rows, ignore_index=True)
        icc_df.columns = ["targets", "raters", "ratings"]
        icc_result = pg.intraclass_corr(
            data=icc_df, targets="targets", raters="raters", ratings="ratings"
        )
        icc1 = icc_result[icc_result["Type"] == "ICC1"]
        val = icc1["ICC"].values[0]
        ci_lo = icc1["CI95%"].values[0][0]
        ci_hi = icc1["CI95%"].values[0][1]
        p_val = icc1["pval"].values[0]
        print(f"  {m}: ICC(1,1) = {val:.3f}  95% CI [{ci_lo:.3f}, {ci_hi:.3f}]  p = {p_val:.4f}")
else:
    print("  Need >= 2 judge runs for ICC")

# ═══════════════════════════════════════════════════════════════════════
# TABLE 6: Tool Usage
# ═══════════════════════════════════════════════════════════════════════
if "tool_used" in all_runs[0].columns:
    print(f"\n{'=' * 70}")
    print("TABLE 6: Tool Usage Distribution (Our RAG)")
    print("=" * 70)
    our_rag_rows = all_runs[0][all_runs[0]["pipeline"] == "our_rag"]
    if not our_rag_rows.empty:
        tool_dist = our_rag_rows["tool_used"].value_counts()
        for tool, count in tool_dist.items():
            print(f"  {tool}: {count} ({count/len(our_rag_rows)*100:.0f}%)")

# ═══════════════════════════════════════════════════════════════════════
# TABLE 7: Ablation (Δ from Our RAG full)
# ═══════════════════════════════════════════════════════════════════════
our_rag_sub = [run[run["pipeline"] == "our_rag"] for run in all_runs]
if not our_rag_sub[0].empty:
    print(f"\n{'=' * 70}")
    print("TABLE 7: Ablation Study (Δ from Our RAG full)")
    print("=" * 70)

    full_scores = {m: np.mean([s[m].mean() for s in our_rag_sub]) for m in metrics}

    abl_rows = []
    for label in ["Our RAG"] + ablation_labels:
        sub_runs = [run[run["pipeline_label"] == label] for run in all_runs]
        if sub_runs[0].empty:
            continue
        row_data = {"variant": label}
        for m in metrics:
            mean_val = np.mean([s[m].mean() for s in sub_runs])
            delta = mean_val - full_scores[m]
            row_data[m] = f"{mean_val:.2f}"
            row_data[f"Δ_{m}"] = "—" if label == "Our RAG" else f"{delta:+.2f}"
        row_data["latency"] = f"{sub_runs[0]['latency_s'].mean():.2f}"
        abl_rows.append(row_data)

    abl_table = pd.DataFrame(abl_rows).set_index("variant")
    print(abl_table.to_markdown())
    abl_table.to_csv("output/ablation_table.csv", encoding="utf-8-sig")

# ── Save ──
table1.to_csv("output/summary_table.csv", encoding="utf-8-sig")
stat_df.to_csv("output/statistical_tests.csv", index=False, encoding="utf-8-sig")
print(f"\nSaved output/summary_table.csv, output/statistical_tests.csv")